---
# Fase 3 · Preparación y modelado de datos

## 5. Configuración del entorno

El proyecto evita dependencias que no estén garantizadas en el entorno de entrega.
Los gráficos del notebook son **SVG generados con la librería propia
`src/graficos_svg.py`** (no requiere matplotlib, seaborn ni plotly) y las pruebas
estadísticas están implementadas en `src/analisis_retail.py`, que reproduce las
distribuciones t, chi-cuadrado y F mediante la función beta/gamma incompleta
regularizada. 

In [1]:
# ---------------------------------------------------------------------------
# Configuración, rutas e importaciones
# ---------------------------------------------------------------------------
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Respaldo de visualización: el notebook funciona con o sin IPython instalado.
try:
    from IPython.display import SVG, display
    _TIENE_IPYTHON = True
except ImportError:  # pragma: no cover
    _TIENE_IPYTHON = False

    def display(*args, **kwargs):
        pass

    class SVG:  # marcador de posición para poder usar display(SVG(...))
        def __init__(self, datos):
            self.datos = datos

RAIZ = Path.cwd()
if not (RAIZ / "src").exists() and (RAIZ.parent / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import graficos_svg as g   # librería de gráficos SVG del proyecto

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RUTA_DATOS = RAIZ / "datos"
RUTA_REPORTES = RAIZ / "reports"
RUTA_GRAFICOS = RAIZ / "graficos"
RUTA_GRAFICOS.mkdir(exist_ok=True)
DIR_ENTREGA = RAIZ / "entrega"
DIR_ENTREGA.mkdir(exist_ok=True)


print(f"IPython disponible: {_TIENE_IPYTHON}")
print(f"pandas {pd.__version__} · numpy {np.__version__}")


def mostrar(svg: str, nombre: str | None = None) -> None:
    """Renderiza un SVG en el notebook y, si se indica, lo exporta a graficos/."""
    if nombre:
        (RUTA_GRAFICOS / f"{nombre}.svg").write_text(svg, encoding="utf-8")
    if _TIENE_IPYTHON:
        display(SVG(svg))
    else:
        print(f"[gráfico {nombre or 'sin nombre'}: {len(svg):,} caracteres SVG]")


def tabla_md(df: pd.DataFrame, dec: int = 2) -> str:
    """Convierte un DataFrame en tabla Markdown (sin depender de tabulate)."""
    enc = [str(c) for c in df.columns]
    filas = []
    for _, r in df.iterrows():
        vals = []
        for v in r:
            if isinstance(v, (int, np.integer)):
                vals.append(f"{v:,}")
            elif isinstance(v, (float, np.floating)):
                vals.append(f"{v:,.{dec}f}")
            else:
                vals.append(str(v))
        filas.append(vals)
    lineas = ["| " + " | ".join(enc) + " |",
              "|" + "|".join(["---"] * len(enc)) + "|"]
    lineas += ["| " + " | ".join(f) + " |" for f in filas]
    return "\n".join(lineas)

IPython disponible: True
pandas 3.0.6 · numpy 2.5.3


## 6. Carga, auditoría y limpieza

La auditoría se documenta como lista de comprobaciones explícitas: cada problema
detectado se registra y cada solución se justifica. Este registro es el insumo del
diagrama de flujo de limpieza que se reporta más abajo.

In [2]:
# ---------------------------------------------------------------------------
# 6.1 Carga y auditoría de calidad
# ---------------------------------------------------------------------------
df_crudo = pd.read_csv(RAIZ / "train.csv", encoding="utf-8", encoding_errors="replace")
df_crudo.columns = [c.strip() for c in df_crudo.columns]

def perfilar_arreglo(a):
    """Devuelve el arreglo de metadatos que muestra el explorador de variables."""
    return pd.DataFrame({
        "tipo": a.dtypes.astype(str),
        "no_nulos": len(a) - a.isna().sum(),
        "nulos": a.isna().sum(),
        "pct_nulos": (a.isna().sum() / len(a) * 100).round(2),
        "unicos": a.nunique(),
        "ejemplo": [a[c].dropna().iloc[0] if a[c].notna().any() else None for c in a.columns],
    })

auditoria = {
    "Filas y columnas": f"{df_crudo.shape[0]:,} filas × {df_crudo.shape[1]} columnas",
    "Columnas presentes": ", ".join(df_crudo.columns),
    "Columnas ausentes relevantes": "Profit, Quantity, Discount (la variante solo trae Sales)",
    "Nulos por columna": ", ".join(f"{k}: {v}" for k, v in df_crudo.isna().sum().items() if v > 0) or "ninguno",
    "Filas duplicadas exactas": int(df_crudo.duplicated().sum()),
    "Duplicados de (Order ID, Product ID)": int(df_crudo.duplicated(subset=["Order ID", "Product ID"]).sum()),
    "Duplicados de Row ID": int(df_crudo.duplicated(subset=["Row ID"]).sum()),
    "Ventas nulas, cero o negativas": int((df_crudo["Sales"].isna() | (df_crudo["Sales"] <= 0)).sum()),
    "Rango de fechas": "2015-01-03 a 2018-12-30",
    "Países distintos": df_crudo["Country"].nunique(),
    "Estados / ciudades": f"{df_crudo['State'].nunique()} / {df_crudo['City'].nunique()}",
}
for k, v in auditoria.items():
    print(f"{k:<40} {v}")

print("\nValores extremos de Sales (¿hay ceros o negativos?):")
print(df_crudo["Sales"].describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_string())

Filas y columnas                         9,800 filas × 18 columnas
Columnas presentes                       Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales
Columnas ausentes relevantes             Profit, Quantity, Discount (la variante solo trae Sales)
Nulos por columna                        Postal Code: 11
Filas duplicadas exactas                 0
Duplicados de (Order ID, Product ID)     8
Duplicados de Row ID                     0
Ventas nulas, cero o negativas           0
Rango de fechas                          2015-01-03 a 2018-12-30
Países distintos                         1
Estados / ciudades                       49 / 529

Valores extremos de Sales (¿hay ceros o negativos?):
count    9,800.0000
mean       230.7700
std        626.6500
min          0.4400
1%           2.2900
50%         54.4900
99%      2,480.2100
max     22,638.4800


In [3]:
# ---------------------------------------------------------------------------
# 6.2 Limpieza y enriquecimiento
# ---------------------------------------------------------------------------
REGISTRO_LIMPIEZA = []

df = df_crudo.copy()

# Tipos de fecha: el archivo usa dd/mm/aaaa, no el formato ISO por defecto.
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="raise")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="raise")
REGISTRO_LIMPIEZA.append(("Fechas en texto dd/mm/aaaa", "Conversión a datetime con dayfirst=True",
                          f"{df['Order Date'].min():%Y-%m-%d} a {df['Order Date'].max():%Y-%m-%d}",
                          "Habilita agregaciones temporales y el cálculo de días de envío"))

# Nulos en Postal Code: 11 registros (0,11 %). No participa en ningún KPI.
nulos_cp = int(df["Postal Code"].isna().sum())
df["Postal Code"] = df["Postal Code"].fillna(0).astype("int64")
REGISTRO_LIMPIEZA.append(("Postal Code con nulos", "Imputación con 0 y conversión a entero",
                          f"{nulos_cp} registros",
                          "Evita errores de tipo sin descartar filas; la variable no se usa en KPIs"))

# Normalización de texto en clasificadores.
antes_texto = df[["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]].copy()
for c in ["Ship Mode", "Segment", "Region", "Category", "Sub-Category", "State", "City"]:
    df[c] = df[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
REGISTRO_LIMPIEZA.append(("Espacios redundantes en clasificadores", "strip + colapso de espacios",
                          f"{df.shape[0]} filas revisadas",
                          "Evita categorías duplicadas por diferencias invisibles de texto"))

# Variable derivada: duración logística.
df["DiasEnvio"] = (df["Ship Date"] - df["Order Date"]).dt.days
inconsistentes = int((df["DiasEnvio"] < 0).sum())
REGISTRO_LIMPIEZA.append(("Duración del envío no existía", "DiasEnvio = Ship Date − Order Date",
                          f"{df['DiasEnvio'].mean():.2f} días en promedio",
                          f"Permite auditar el desempeño logístico; inconsistencias: {inconsistentes}"))

# Variables de calendario y de segmentación de valor.
df["Anio"] = df["Order Date"].dt.year
df["Trimestre"] = df["Order Date"].dt.quarter
df["Mes"] = df["Order Date"].dt.month
df["MesNombre"] = df["Mes"].map(dict(enumerate(g.__dict__.get("MESES", []) or
    ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"], 1)))
df["AnioMes"] = df["Order Date"].dt.to_period("M").astype(str)
df["AnioTrimestre"] = df["Order Date"].dt.to_period("Q").astype(str)
df["DiaSemana"] = df["Order Date"].dt.dayofweek
df["EsQ4"] = df["Mes"].isin([10, 11, 12])
df["TramoLinea"] = pd.cut(df["Sales"], [0, 50, 150, 400, 1000, np.inf],
                          labels=["<50", "50-150", "150-400", "400-1000", ">1000"])
REGISTRO_LIMPIEZA.append(("Sin dimensiones temporales ni tramos", "Se crean 8 columnas derivadas",
                          "Anio, Trimestre, Mes, AnioMes, AnioTrimestre, DiaSemana, EsQ4, TramoLinea",
                          "Sostienen la estacionalidad, la comparabilidad y el test de H5"))

# Validación cruzada de identificadores con la tabla de pedidos.
ped_control = df.groupby("Order ID").agg(ventas=("Sales", "sum"), lineas=("Row ID", "count"))
assert ped_control["ventas"].gt(0).all(), "Hay pedidos con venta no positiva"

registro_df = pd.DataFrame(REGISTRO_LIMPIEZA,
                           columns=["Problema o necesidad", "Tratamiento", "Magnitud", "Justificación"])
print(f"Registro de limpieza: {len(registro_df)} intervenciones documentadas")
print(f"Filas conservadas: {len(df):,} de {len(df_crudo):,} "
      f"({len(df)/len(df_crudo)*100:.2f} %) — no se eliminó ningún registro")
registro_df

Registro de limpieza: 5 intervenciones documentadas
Filas conservadas: 9,800 de 9,800 (100.00 %) — no se eliminó ningún registro


,Problema o necesidad,Tratamiento,Magnitud,Justificación
0,Fechas en texto dd/mm/aaaa,Conversión a datetime con dayfirst=True,2015-01-03 a 2018-12-30,Habilita agregaciones temporales y el cálculo ...
1,Postal Code con nulos,Imputación con 0 y conversión a entero,11 registros,Evita errores de tipo sin descartar filas; la ...
2,Espacios redundantes en clasificadores,strip + colapso de espacios,9800 filas revisadas,Evita categorías duplicadas por diferencias in...
3,Duración del envío no existía,DiasEnvio = Ship Date − Order Date,3.96 días en promedio,Permite auditar el desempeño logístico; incons...
4,Sin dimensiones temporales ni tramos,Se crean 8 columnas derivadas,"Anio, Trimestre, Mes, AnioMes, AnioTrimestre, ...","Sostienen la estacionalidad, la comparabilidad..."


In [4]:
# ---------------------------------------------------------------------------
# 6.3 Verificación posterior a la limpieza
# ---------------------------------------------------------------------------
controles = pd.DataFrame([
    ("Sin nulos tras la limpieza", int(df.isna().sum().sum()), "0"),
    ("Tipos de fecha correctos", str(df[["Order Date", "Ship Date"]].dtypes.unique().tolist()),
     "datetime64[ns]"),
    ("Sin ventas ≤ 0", int((df["Sales"] <= 0).sum()), "0"),
    ("Sin duplicados de línea (Row ID)", int(df.duplicated(subset=["Row ID"]).sum()), "0"),
    ("Pedidos reconstruidos", int(df["Order ID"].nunique()), "4.922"),
    ("Días de envío negativos", int((df["DiasEnvio"] < 0).sum()), "0"),
    ("Rango de días de envío", f"{int(df['DiasEnvio'].min())} a {int(df['DiasEnvio'].max())}", "0 a 7"),
    ("Categorías distintas", int(df["Category"].nunique()), "3"),
    ("Subcategorías distintas", int(df["Sub-Category"].nunique()), "17"),
], columns=["Control", "Resultado", "Esperado"])
print(controles.to_string(index=False))

                         Control          Resultado       Esperado
      Sin nulos tras la limpieza                  0              0
        Tipos de fecha correctos [dtype('<M8[us]')] datetime64[ns]
                  Sin ventas ≤ 0                  0              0
Sin duplicados de línea (Row ID)                  0              0
           Pedidos reconstruidos               4922          4.922
         Días de envío negativos                  0              0
          Rango de días de envío              0 a 7          0 a 7
            Categorías distintas                  3              3
         Subcategorías distintas                 17             17


## 7. Selección de variables y modelo de datos

| Variable | Rol | Tipo | Uso en el análisis |
|---|---|---|---|
| `Sales` | Medida | Continua (USD) | Único hecho del dataset: todos los KPIs de ingreso |
| `Order Date` | Dimensión tiempo | Fecha | Tendencia, estacionalidad, comparación interanual |
| `Ship Date` / `DiasEnvio` | Dimensión operativa | Fecha / entero | KPI logístico y H5 |
| `Category`, `Sub-Category` | Dimensión producto | Categórica | Mezcla, Pareto y treemap |
| `Product Name` | Dimensión producto | Categórica (1.861) | Ranking y concentración de catálogo |
| `Region`, `State`, `City` | Dimensión geografía | Categórica | H2, mapa y cuota territorial |
| `Segment` | Dimensión cliente | Categórica (3) | Ticket y mix por tipo de comprador |
| `Ship Mode` | Dimensión logística | Categórica (4) | H5 y nivel de servicio |
| `Customer ID` | Dimensión cliente | Categórica (793) | Frecuencia, recurrencia y Gini de clientes |
| `Postal Code` | Atributo | Entero | Solo llave geográfica auxiliar |
| `Row ID`, `Order ID` | Llaves | Entero / texto | Grano y agregación a pedido |
| `Country` | Descartada | Constante | Un solo valor: no aporta información |

**Modelo de datos.** El análisis trabaja en dos granos, porque las preguntas operan
en niveles distintos:

- **Grano línea de pedido** (`datos/analitica_ventas_linea.csv`, 9.800 filas): mezcla de
  producto, geografía y logística.
- **Grano pedido** (`datos/analitica_pedidos.csv`, 4.922 filas): ticket, frecuencia y
  valor de cliente. Es el grano correcto para todo lo relacionado con el comprador.

**Esquema en estrella** para el dashboard (implementado en Power BI según la
especificación técnica): tabla de hechos `FactVentas` (grano línea, medida `Sales`)
con dimensiones `DimCalendario`, `DimProducto`, `DimCliente`, `DimGeografia` y
`DimEnvio`. El detalle de relaciones, granularidad y medidas DAX está en
`docs/especificacion_power_bi.md`.

In [5]:
# ---------------------------------------------------------------------------
# 7.1 Construcción de las tablas de análisis (grano línea y grano pedido)
# ---------------------------------------------------------------------------
pedidos = (df.groupby("Order ID")
           .agg(Ventas=("Sales", "sum"),
                Lineas=("Row ID", "count"),
                Productos=("Product ID", "nunique"),
                Cliente=("Customer ID", "first"),
                Segmento=("Segment", "first"),
                Region=("Region", "first"),
                Estado=("State", "first"),
                Ciudad=("City", "first"),
                ModoEnvio=("Ship Mode", "first"),
                Fecha=("Order Date", "min"),
                DiasEnvio=("DiasEnvio", "max"))
           .reset_index())
pedidos["Anio"] = pedidos["Fecha"].dt.year
pedidos["AnioTrimestre"] = pedidos["Fecha"].dt.to_period("Q").astype(str)
pedidos["EsQ4"] = pedidos["Fecha"].dt.month.isin([10, 11, 12])
pedidos["TramoPedido"] = pd.cut(pedidos["Ventas"], [0, 100, 250, 500, 1000, np.inf],
                                labels=["<100", "100-250", "250-500", "500-1000", ">1000"])

print(f"Grano línea : {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Grano pedido: {pedidos.shape[0]:,} filas × {pedidos.shape[1]} columnas")
print()
resumen_grano = pd.DataFrame([
    ("Ingreso total", f"{df['Sales'].sum():,.2f}", f"{pedidos['Ventas'].sum():,.2f}"),
    ("Media", f"{df['Sales'].mean():,.2f}", f"{pedidos['Ventas'].mean():,.2f}"),
    ("Mediana", f"{df['Sales'].median():,.2f}", f"{pedidos['Ventas'].median():,.2f}"),
    ("Mínimo", f"{df['Sales'].min():,.2f}", f"{pedidos['Ventas'].min():,.2f}"),
    ("Máximo", f"{df['Sales'].max():,.2f}", f"{pedidos['Ventas'].max():,.2f}"),
], columns=["Estadístico", "Línea de pedido", "Pedido"])
print(resumen_grano.to_string(index=False))
print(f"\nLíneas por pedido: {pedidos['Lineas'].mean():.3f} en promedio "
      f"(máximo {int(pedidos['Lineas'].max())})")

Grano línea : 9,800 filas × 28 columnas
Grano pedido: 4,922 filas × 16 columnas

  Estadístico Línea de pedido       Pedido
Ingreso total    2,261,536.78 2,261,536.78
        Media          230.77       459.48
      Mediana           54.49       151.88
       Mínimo            0.44         0.56
       Máximo       22,638.48    23,661.23

Líneas por pedido: 1.991 en promedio (máximo 14)


In [6]:
# ---------------------------------------------------------------------------
# 7.2 Exploración inicial: calidad, cardinalidad y forma de la distribución
# ---------------------------------------------------------------------------
print("Distribución del valor por línea y por pedido")
desc = pd.DataFrame({
    "Línea de pedido": df["Sales"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]),
    "Pedido": pedidos["Ventas"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]),
}).round(2)
print(desc.to_string())

asimetria_pedido = float(pedidos["Ventas"].skew())
print(f"\nAsimetría del valor de pedido: {asimetria_pedido:.3f} "
      f"→ distribución fuertemente sesgada a la derecha.")
print("Consecuencia metodológica: se prefieren la mediana y las pruebas de Welch sobre "
      "el logaritmo, y se reportan intervalos de confianza en lugar de solo promedios.")

cardinalidad = pd.DataFrame([
    ("Order ID", df["Order ID"].nunique(), "grano de pedido"),
    ("Customer ID", df["Customer ID"].nunique(), "padrón de clientes"),
    ("Product ID", df["Product ID"].nunique(), "catálogo"),
    ("Product Name", df["Product Name"].nunique(), "catálogo (etiqueta)"),
    ("City", df["City"].nunique(), "cobertura urbana"),
    ("State", df["State"].nunique(), "cobertura estatal"),
], columns=["Entidad", "Valores distintos", "Lectura"])
print()
print(cardinalidad.to_string(index=False))

Distribución del valor por línea y por pedido
       Línea de pedido      Pedido
count       9,800.0000  4,922.0000
mean          230.7700    459.4800
std           626.6500    958.6700
min             0.4400      0.5600
25%            17.2500     37.2700
50%            54.4900    151.8800
75%           210.6000    513.5100
90%           575.9300  1,158.4200
99%         2,480.2100  4,206.3600
max        22,638.4800 23,661.2300

Asimetría del valor de pedido: 8.348 → distribución fuertemente sesgada a la derecha.
Consecuencia metodológica: se prefieren la mediana y las pruebas de Welch sobre el logaritmo, y se reportan intervalos de confianza en lugar de solo promedios.

     Entidad  Valores distintos             Lectura
    Order ID               4922     grano de pedido
 Customer ID                793  padrón de clientes
  Product ID               1861            catálogo
Product Name               1849 catálogo (etiqueta)
        City                529    cobertura urbana
       St

In [7]:
# ---------------------------------------------------------------------------
# 7.3 Exportación de tablas limpias para el dashboard
# ---------------------------------------------------------------------------
tablas_entrega = {
    "analitica_ventas_linea": df,
    "analitica_pedidos": pedidos,
}
for nombre, t in tablas_entrega.items():
    (RUTA_DATOS / f"{nombre}_notebook.csv").parent.mkdir(parents=True, exist_ok=True)
    t.to_csv(RUTA_DATOS / f"{nombre}_notebook.csv", index=False, encoding="utf-8-sig")
print("Tablas exportadas a datos/:")
for nombre, t in tablas_entrega.items():
    print(f"  {nombre + '_notebook.csv':<38} {t.shape[0]:>6,} × {t.shape[1]:>2}")
print("\nLas tablas definitivas del pipeline (incluidas las dimensiones del esquema en "
      "estrella) se generan con:\n    python src/analisis_retail.py")

Tablas exportadas a datos/:
  analitica_ventas_linea_notebook.csv     9,800 × 28
  analitica_pedidos_notebook.csv          4,922 × 16

Las tablas definitivas del pipeline (incluidas las dimensiones del esquema en estrella) se generan con:
    python src/analisis_retail.py
